In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Row
import uuid
from datetime import datetime

In [0]:
asset_health = spark.table(
    "platform_monitoring.asset_health"
)

job_health = spark.table(
    "platform_monitoring.job_health"
)

In [0]:
violations = []
stale_tables = asset_health.filter(
    F.col("days_unused") > 90
)

for row in stale_tables.collect():

    violations.append(

        Row(

            violation_id=str(uuid.uuid4()),

            violation_type="STALE_TABLE",

            object_type="TABLE",

            object_name=row.table_name,

            severity="HIGH",

            recommendation="Archive or delete unused table",

            detected_time=datetime.now(),

            status="OPEN"

        )
    )

In [0]:
warning_tables = asset_health.filter(

    (F.col("days_unused") >= 30)

    &

    (F.col("days_unused") <= 90)

)

for row in warning_tables.collect():

    violations.append(

        Row(

            violation_id=str(uuid.uuid4()),

            violation_type="LOW_USAGE",

            object_type="TABLE",

            object_name=row.table_name,

            severity="MEDIUM",

            recommendation="Review business usage",

            detected_time=datetime.now(),

            status="OPEN"

        )

    )

In [0]:
critical_jobs = job_health.filter(
    F.col("success_rate") < 70
)

for row in critical_jobs.collect():

    violations.append(

        Row(

            violation_id=str(uuid.uuid4()),

            violation_type="LOW_SUCCESS_RATE",

            object_type="JOB",

            object_name=row.job_name,

            severity="HIGH",

            recommendation="Investigate recurring failures",

            detected_time=datetime.now(),
            status="OPEN"

        )

    )

In [0]:
warning_jobs = job_health.filter(

    (F.col("success_rate") >= 70)

    &

    (F.col("success_rate") < 90)

)

for row in warning_jobs.collect():

    violations.append(

        Row(

            violation_id=str(uuid.uuid4()),

            violation_type="LOW_SUCCESS_RATE",

            object_type="JOB",

            object_name=row.job_name,

            severity="MEDIUM",

            recommendation="Monitor job health",

            detected_time=datetime.now(),

            status="OPEN"

        )

    )

In [0]:
slow_jobs = job_health.filter(
    F.col("avg_runtime") > 45
)

for row in slow_jobs.collect():

    violations.append(

        Row(

            violation_id=str(uuid.uuid4()),

            violation_type="SLOW_JOB",

            object_type="JOB",

            object_name=row.job_name,

            severity="LOW",

            recommendation="Optimize Spark execution",

            detected_time=datetime.now(),

            status="OPEN"

        )

    )

In [0]:

violations_df = spark.createDataFrame(
    violations
)

In [0]:
violations_df = (

    violations_df

    .withColumn(

        "detected_time",

        F.current_timestamp()

    )
)

In [0]:
violations_df.write \
.mode("overwrite") \
.format("delta") \
.saveAsTable(
"platform_monitoring.governance_violations"
)

In [0]:
spark.table(
"platform_monitoring.governance_violations"
).show(
50,
False
)

+------------------------------------+----------------+-----------+------------------+--------+------------------------------+--------------------------+------+
|violation_id                        |violation_type  |object_type|object_name       |severity|recommendation                |detected_time             |status|
+------------------------------------+----------------+-----------+------------------+--------+------------------------------+--------------------------+------+
|bfdc132a-a4d1-4254-8199-9ea5bbef4e1d|STALE_TABLE     |TABLE      |audit_logs        |HIGH    |Archive or delete unused table|2026-06-30 17:21:28.805077|OPEN  |
|3ec72c10-08ee-42a1-9be1-44829a0158ea|LOW_USAGE       |TABLE      |shipments         |MEDIUM  |Review business usage         |2026-06-30 17:21:28.805077|OPEN  |
|13df8a3c-9500-406f-b039-495ced82b1a3|LOW_SUCCESS_RATE|JOB        |Inventory_Sync    |HIGH    |Investigate recurring failures|2026-06-30 17:21:28.805077|OPEN  |
|00b0abc7-3f13-4bd1-a011-bd376f24f